In [2]:
import pandas as pd
import numpy as np

# ========== Helper function: find the SMILES column ==========
def find_smiles_column(df):
    """Find the SMILES column in a dataframe (case-insensitive)"""
    possible_names = ['SMILES', 'Smiles', 'smiles', 'Canonical SMILES']
    for col in df.columns:
        if col in possible_names:
            return col
    raise ValueError(f"No SMILES column found. Available columns: {df.columns.tolist()}")

# ========== 1. Read data ==========
print("Loading datasets...")
pkis = pd.read_csv('/home/s1856122/ddm/data_source/PKIS/PKIS1_2_comprehension_with_uniprot_normalized_inh.csv')
dude = pd.read_csv('/home/s1856122/ddm/data_source/DUDE/dude_kinase_full.csv')
dudz = pd.read_csv('/home/s1856122/ddm/data_source/DUDE-Z/DUDZ_integrated_data.csv')
davis = pd.read_csv('/home/s1856122/ddm/data_source/davis/davis_comprehension_no_duplicate.csv')
vsds = pd.read_csv('/home/s1856122/ddm/data_source/VSDS/VSDS_truedecoy_integrated_data_with_protein_seq.csv')

# ========== 2. Process each dataset ==========
print("\nProcessing individual datasets...")

# PKIS
pkis_smiles_col = find_smiles_column(pkis)
print(f"PKIS SMILES column: {pkis_smiles_col}")
pkis_processed = pkis[[pkis_smiles_col, 'Morgan', 'KLIFS', 'Bioactivity']].copy()
pkis_processed['unified_bio_label'] = (pkis_processed['Bioactivity'] > 50).astype(int)
pkis_processed = pkis_processed[[pkis_smiles_col, 'Morgan', 'KLIFS', 'unified_bio_label']]
pkis_processed = pkis_processed.rename(columns={pkis_smiles_col: 'SMILES'})
pkis_processed['source'] = 'PKIS'
print(f"PKIS: {len(pkis_processed)} rows")

# DUDE
dude_smiles_col = find_smiles_column(dude)
print(f"DUDE SMILES column: {dude_smiles_col}")
dude_processed = dude[[dude_smiles_col, 'Morgan', 'KLIFS', 'molecule_label']].copy()
dude_processed = dude_processed.rename(columns={
    dude_smiles_col: 'SMILES',
    'molecule_label': 'unified_bio_label'
})
dude_processed['source'] = 'DUDE'
print(f"DUDE: {len(dude_processed)} rows")

# DUDE-Z
dudz_smiles_col = find_smiles_column(dudz)
print(f"DUDE-Z SMILES column: {dudz_smiles_col}")
dudz_processed = dudz[[dudz_smiles_col, 'Morgan', 'KLIFS', 'molecule_label']].copy()
dudz_processed = dudz_processed.rename(columns={
    dudz_smiles_col: 'SMILES',
    'molecule_label': 'unified_bio_label'
})
dudz_processed['source'] = 'DUDZ'
print(f"DUDE-Z: {len(dudz_processed)} rows")

# Davis
davis_smiles_col = find_smiles_column(davis)
print(f"Davis SMILES column: {davis_smiles_col}")
davis_processed = davis[[davis_smiles_col, 'Morgan', 'KLIFS', 'Bioactivity']].copy()
davis_processed['unified_bio_label'] = (davis_processed['Bioactivity'] < 1000).astype(int)
davis_processed = davis_processed[[davis_smiles_col, 'Morgan', 'KLIFS', 'unified_bio_label']]
davis_processed = davis_processed.rename(columns={davis_smiles_col: 'SMILES'})
davis_processed['source'] = 'Davis'
print(f"Davis: {len(davis_processed)} rows")

# VSDS (reference set - read only, do not modify)
vsds_smiles_col = find_smiles_column(vsds)
print(f"VSDS SMILES column: {vsds_smiles_col}")
vsds_ref = vsds[[vsds_smiles_col, 'Morgan', 'KLIFS', 'molecule_label']].copy()
vsds_ref = vsds_ref.rename(columns={
    vsds_smiles_col: 'SMILES',
    'molecule_label': 'unified_bio_label'
})
print(f"VSDS (reference): {len(vsds_ref)} rows")

# ========== 3. Merge the four datasets ==========
merged_data = pd.concat([
    pkis_processed,
    dude_processed,
    dudz_processed,
    davis_processed
], ignore_index=True)

print(f"\nTotal merged rows: {len(merged_data)}")
print(f"Active: {(merged_data['unified_bio_label'] == 1).sum()}")
print(f"Inactive: {(merged_data['unified_bio_label'] == 0).sum()}")

# ========== 4. Internal deduplication - handle conflicts ==========
print("\n" + "="*60)
print("Step 1: Internal deduplication")
print("="*60)

# Create composite keys
merged_data['key'] = merged_data['Morgan'] + '|||' + merged_data['KLIFS']

# Find duplicate keys
duplicated_keys = merged_data[merged_data.duplicated(subset='key', keep=False)].copy()

internal_conflicts_list = []
keys_to_remove = set()

if len(duplicated_keys) > 0:
    print(f"Found {duplicated_keys['key'].nunique()} duplicated Morgan+KLIFS combinations")
    
    for key, group in duplicated_keys.groupby('key'):
        unique_labels = group['unified_bio_label'].unique()
        if len(unique_labels) > 1:
            # Label conflict - record all related rows and mark them for removal
            internal_conflicts_list.append(group)
            keys_to_remove.add(key)
            print(f"⚠️  Internal conflict: {key[:50]}... has labels {unique_labels.tolist()}")

# Save internal conflicts
if internal_conflicts_list:
    internal_conflicts = pd.concat(internal_conflicts_list, ignore_index=True)
    internal_conflicts = internal_conflicts.drop('key', axis=1)
    conflict_path = '/home/s1856122/ddm/SPECTRA/ready_to_use/internal_conflicts.csv'
    internal_conflicts.to_csv(conflict_path, index=False)
    print(f"\n❌ {len(internal_conflicts)} conflicting rows saved to: {conflict_path}")
    print(f"   ({len(keys_to_remove)} unique Morgan+KLIFS combinations with conflicts)")
    
    # Remove all conflicting keys from merged
    merged_no_conflicts = merged_data[~merged_data['key'].isin(keys_to_remove)].copy()
    print(f"✓ Removed all conflicting entries from merged data")
else:
    print("✓ No internal conflicts found")
    merged_no_conflicts = merged_data.copy()

# Deduplicate the remaining data (keep the first record)
merged_dedup = merged_no_conflicts.drop_duplicates(subset='key', keep='first').copy()
print(f"✓ After internal dedup: {len(merged_dedup)} rows (removed {len(merged_no_conflicts) - len(merged_dedup)} exact duplicates)")

# ========== 5. Deduplicate against VSDS ==========
print("\n" + "="*60)
print("Step 2: Deduplication against VSDS")
print("="*60)

# Create keys for VSDS
vsds_ref['key'] = vsds_ref['Morgan'] + '|||' + vsds_ref['KLIFS']

# Find records duplicated with VSDS
merged_with_vsds = merged_dedup.merge(
    vsds_ref[['key', 'unified_bio_label']], 
    on='key', 
    how='left', 
    suffixes=('', '_vsds')
)

# Find overlapping records
vsds_overlap = merged_with_vsds[merged_with_vsds['unified_bio_label_vsds'].notna()].copy()

vsds_conflicts_list = []
keys_to_remove_vsds = set()

if len(vsds_overlap) > 0:
    print(f"Found {len(vsds_overlap)} overlapping entries with VSDS")
    
    # Check label consistency
    vsds_overlap['label_match'] = (
        vsds_overlap['unified_bio_label'] == vsds_overlap['unified_bio_label_vsds']
    )
    
    consistent = vsds_overlap[vsds_overlap['label_match'] == True]
    conflicts = vsds_overlap[vsds_overlap['label_match'] == False]
    
    print(f"  - Consistent labels: {len(consistent)} (will be removed)")
    print(f"  - Conflicting labels: {len(conflicts)} (will be removed and saved)")
    
    # Save conflicts with VSDS
    if len(conflicts) > 0:
        conflicts_clean = conflicts.drop(['key', 'unified_bio_label_vsds', 'label_match'], axis=1)
        vsds_conflict_path = '/home/s1856122/ddm/SPECTRA/ready_to_use/vsds_conflicts.csv'
        conflicts_clean.to_csv(vsds_conflict_path, index=False)
        print(f"\n❌ {len(conflicts)} VSDS conflicts saved to: {vsds_conflict_path}")
        
        # Show conflict examples
        print("\nSample VSDS conflicts:")
        for idx, (_, row) in enumerate(conflicts.head(3).iterrows(), 1):
            print(f"  [{idx}] Source: {row['source']}")
            print(f"      Merged label: {row['unified_bio_label']}, VSDS label: {int(row['unified_bio_label_vsds'])}")
            print(f"      SMILES: {row['SMILES'][:40]}...")
            print(f"      Morgan: {row['Morgan'][:40]}...\n")
    
    # Remove all records duplicated with VSDS (both consistent and conflicting records)
    all_vsds_keys = vsds_overlap['key'].unique()
    final_merged = merged_with_vsds[~merged_with_vsds['key'].isin(all_vsds_keys)].copy()
    final_merged = final_merged.drop(['key', 'unified_bio_label_vsds'], axis=1)
    
    print(f"✓ Removed all {len(vsds_overlap)} VSDS overlaps from merged data")
else:
    print("✓ No overlap with VSDS found")
    final_merged = merged_dedup.drop('key', axis=1)

# ========== 6. Save final results ==========
print("\n" + "="*60)
print("Final Results")
print("="*60)

# Adjust column order: SMILES, Morgan, KLIFS, unified_bio_label, source
final_merged = final_merged[['SMILES', 'Morgan', 'KLIFS', 'unified_bio_label', 'source']]

output_path = '/home/s1856122/ddm/SPECTRA/ready_to_use/train_ready2use_merged_4datasets_deduplicated_with_smiles.csv'
final_merged.to_csv(output_path, index=False)

print(f"\n✓ Final clean merged data: {len(final_merged)} rows")
print(f"  - Active: {(final_merged['unified_bio_label'] == 1).sum()}")
print(f"  - Inactive: {(final_merged['unified_bio_label'] == 0).sum()}")
print(f"\nSaved to: {output_path}")

# Data-source statistics
print("\nData source distribution:")
print(final_merged['source'].value_counts())

# Show examples from the first few rows
print("\nFirst few rows:")
print(final_merged.head())

print("\n" + "="*60)
print("Processing Summary")
print("="*60)
print(f"Initial merged rows: {len(merged_data)}")
print(f"After removing internal conflicts: {len(merged_dedup)}")
print(f"After removing VSDS overlaps: {len(final_merged)}")
print(f"Total removed: {len(merged_data) - len(final_merged)}")
if internal_conflicts_list:
    print(f"  - Internal conflicts: {len(pd.concat(internal_conflicts_list))}")
if len(vsds_overlap) > 0:
    print(f"  - VSDS overlaps: {len(vsds_overlap)}")
    print(f"    (including {len(conflicts)} conflicts and {len(consistent)} consistent)")

Loading datasets...

Processing individual datasets...
PKIS SMILES column: SMILES
PKIS: 305594 rows
DUDE SMILES column: smiles
DUDE: 360913 rows
DUDE-Z SMILES column: SMILES
DUDE-Z: 34017 rows
Davis SMILES column: Canonical SMILES
Davis: 24408 rows
VSDS SMILES column: SMILES
VSDS (reference): 49706 rows

Total merged rows: 724932
Active: 31495
Inactive: 693437

Step 1: Internal deduplication
Found 28130 duplicated Morgan+KLIFS combinations
⚠️  Internal conflict: 00000000000000000000000000000000000000000000000000... has labels [1, 0]
⚠️  Internal conflict: 00000000000000000000000000000000000000000000000000... has labels [0, 1]
⚠️  Internal conflict: 00000000000000000000000000000000000000000000000000... has labels [0, 1]
⚠️  Internal conflict: 00000000000000000000000000000000000000000000000000... has labels [0, 1]
⚠️  Internal conflict: 00000000000000000000000000000000000000000000000000... has labels [1, 0]
⚠️  Internal conflict: 00000000000000000000000000000000000000000000000000... has 